# Olasheni's Deal Hunter

An agent that hunts for deals you’d actually want: **RSS → pick promising ones with an LLM → estimate fair price with RAG over my catalog → surface the best.** When it finds something good, send it to my phone.

Inspired by the instructor: *"Have this up and running… take great joy every time you get that text message… your agent has found another compelling deal."*

**What’s mine:**
- **LiteLLM** for all model calls (one interface, any provider).
- **My HuggingFace catalog** (Olasheni) for RAG so pricing is grounded in my data.
- **One “Run hunt”** → one hero deal + table; **“Send to my phone”** for the top pick.

**Env:** `OPENROUTER_API_KEY` or any LiteLLM-supported key; `HF_TOKEN`; optional `PUSHOVER_USER` / `PUSHOVER_TOKEN`.

## 1. Setup: LiteLLM + env

In [12]:
import os
import re
import json
from pathlib import Path
from typing import List, Optional

from dotenv import load_dotenv
from litellm import completion
from pydantic import BaseModel, Field
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import feedparser
from bs4 import BeautifulSoup
import requests
from tqdm.notebook import tqdm
import gradio as gr

load_dotenv(override=True)

# LiteLLM: use OpenRouter with one model for both picking and pricing (simpler, all mine)
HUNT_MODEL = "openrouter/deepseek/deepseek-chat"
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

def llm(prompt: str, system: Optional[str] = None, max_tokens: int = 500) -> str:
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    r = completion(model=HUNT_MODEL, messages=messages, max_tokens=max_tokens)
    return (r.choices[0].message.content or "").strip()

print("HUNT_MODEL:", HUNT_MODEL)
print("Pushover:", "on" if (os.getenv("PUSHOVER_USER") and os.getenv("PUSHOVER_TOKEN")) else "off (optional)")

HUNT_MODEL: openrouter/deepseek/deepseek-chat
Pushover: off (optional)


## 2. Data shapes

In [13]:
class Deal(BaseModel):
    product_description: str = Field(description="What the product is")
    price: float = Field(description="Listed price")
    url: str = Field(description="Link")

class PickedDeal(BaseModel):
    deals: List[Deal] = Field(description="Up to 5 deals")

class HuntResult(BaseModel):
    deal: Deal
    estimate: float
    discount: float

## 3. My catalog (HuggingFace) + RAG store

In [14]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=False)

# My catalog (or fallback to course data)
CATALOG_USER = "Olasheni"
LITE = True
try:
    ds = load_dataset(f"{CATALOG_USER}/items_lite" if LITE else f"{CATALOG_USER}/items_full")
except Exception:
    ds = load_dataset("ed-donner/items_lite" if LITE else "ed-donner/items_full")
    print("Using ed-donner catalog (set Olasheni dataset on Hub to use mine)")
rows = list(ds["train"])

def doc_text(r):
    return (r.get("summary") or r.get("title") or "")[:500]

print("Catalog rows:", len(rows))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Using ed-donner catalog (set Olasheni dataset on Hub to use mine)
Catalog rows: 20000


In [15]:
NOTEBOOK_DIR = Path.cwd()
DB_PATH = NOTEBOOK_DIR / "olasheni_deal_hunter_db"
DB_PATH.mkdir(exist_ok=True)

chroma = chromadb.PersistentClient(path=str(DB_PATH))
coll_name = "catalog"
if coll_name in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(coll_name)
coll = chroma.create_collection(coll_name)

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
CAP = 20_000
BATCH = 1000

for i in tqdm(range(0, min(CAP, len(rows)), BATCH)):
    batch = rows[i : i + BATCH]
    docs = [doc_text(r) for r in batch]
    vecs = encoder.encode(docs).astype(float).tolist()
    metas = [{"price": float(r.get("price", 0))} for r in batch]
    ids = [f"d_{i+j}" for j in range(len(batch))]
    coll.add(ids=ids, documents=docs, embeddings=vecs, metadatas=metas)

print("RAG collection count:", coll.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  0%|          | 0/20 [00:00<?, ?it/s]

RAG collection count: 20000


## 4. RSS → raw deals

In [ ]:
FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
]

def fetch_rss_deals(limit_per_feed: int = 8) -> List[dict]:
    out = []
    for url in FEEDS:
        feed = feedparser.parse(url)
        for e in feed.entries[:limit_per_feed]:
            title = (e.get("title") or "")[:120]
            summary = (e.get("summary") or "")
            if hasattr(summary, "replace"):
                summary = BeautifulSoup(summary, "html.parser").get_text(strip=True)[:400]
            link = (e.get("links", [{}])[0].get("href", ""))
            out.append({"title": title, "summary": summary, "url": link})
    return out

sample = fetch_rss_deals(limit_per_feed=2)
print(f"Fetched {len(sample)} sample entries")

## 5. LLM: pick up to 5 deals with clear prices

In [ ]:
PICK_SYSTEM = """You pick the best deals: clear product description and a real numeric price in USD.
Reply with ONLY valid JSON, no markdown: {"deals": [{"product_description": "...", "price": 99.99, "url": "https://..."}, ...]}.
At most 5 deals. Skip any where price is missing or unclear."""

def pick_deals_with_llm(raw: List[dict]) -> List[Deal]:
    if not raw:
        return []
    text = "\n\n".join([f"Title: {r['title']}\nSummary: {r['summary']}\nURL: {r['url']}" for r in raw[:25]])
    out = llm("Deals from RSS:\n\n" + text + "\n\nReturn JSON with key 'deals' (array of objects: product_description, price, url).", system=PICK_SYSTEM, max_tokens=800)
    out = re.sub(r"^```\w*\n?", "", out).rstrip("`")
    try:
        data = json.loads(out)
        return [Deal(**d) for d in data.get("deals", []) if isinstance(d.get("price"), (int, float)) and d.get("price", 0) > 0][:5]
    except Exception:
        return []

## 6. RAG + LLM: estimate fair price

In [ ]:
def estimate_fair_price(description: str, top_k: int = 5) -> float:
    vec = encoder.encode([description])
    res = coll.query(query_embeddings=vec.astype(float).tolist(), n_results=top_k)
    docs = res["documents"][0]
    prices = [m["price"] for m in res["metadatas"][0]]
    context = "Similar products and prices:\n" + "\n".join([f"- {d[:180]}... ${p:.2f}" for d, p in zip(docs, prices)])
    prompt = f"Estimate the fair price in USD for this product. Reply with one number only.\n\nProduct:\n{description}\n\n{context}"
    reply = llm(prompt, max_tokens=20)
    m = re.search(r"[-+]?\d*\.?\d+", reply.replace("$", "").replace(",", ""))
    return float(m.group()) if m else 0.0

def run_hunt(already_seen_urls: Optional[List[str]] = None) -> List[HuntResult]:
    seen = set(already_seen_urls or [])
    raw = [r for r in fetch_rss_deals() if r["url"] not in seen]
    deals = pick_deals_with_llm(raw)
    results = []
    for d in deals:
        est = estimate_fair_price(d.product_description)
        discount = est - d.price if est > 0 else 0.0
        results.append(HuntResult(deal=d, estimate=est, discount=discount))
    results.sort(key=lambda x: x.discount, reverse=True)
    return results

## 7. Send to my phone (Pushover)

In [ ]:
def send_to_phone(result: HuntResult) -> bool:
    u, t = os.getenv("PUSHOVER_USER"), os.getenv("PUSHOVER_TOKEN")
    if not u or not t:
        return False
    msg = f"Deal! ${result.deal.price:.2f} (est ${result.estimate:.2f}, save ${result.discount:.2f}) {result.deal.product_description[:60]}... {result.deal.url}"
    try:
        requests.post(PUSHOVER_URL, data={"user": u, "token": t, "message": msg[:1024], "sound": "cashregister"}, timeout=5)
        return True
    except Exception:
        return False

print("Send to phone: ready" if (os.getenv("PUSHOVER_USER") and os.getenv("PUSHOVER_TOKEN")) else "Set PUSHOVER_USER and PUSHOVER_TOKEN to enable")

## 8. Gradio: Run hunt → hero deal + table → Send to my phone

In [ ]:
def send_best_from_table(results_state, evt: gr.SelectData):
    if not results_state or evt.index[0] >= len(results_state):
        return "Select a row first."
    ok = send_to_phone(results_state[evt.index[0]])
    return "Sent to your phone." if ok else "Pushover not configured."

In [ ]:
def ui_hunt():
    results_state = gr.State([])

    def run_and_store():
        results = run_hunt()
        if not results:
            return "No deals this run.", [], results, ""
        best = results[0]
        table = [[r.deal.product_description[:70] + "...", f"${r.deal.price:.2f}", f"${r.estimate:.2f}", f"${r.discount:.2f}", r.deal.url] for r in results]
        hero = f"**My agent found this for you**\n\n{best.deal.product_description[:200]}...\n\nListed **${best.deal.price:.2f}** · Fair value **${best.estimate:.2f}** · You save **${best.discount:.2f}**"
        return hero, table, results, ""

    with gr.Blocks(title="Olasheni's Deal Hunter", fill_width=True) as ui:
        gr.Markdown("# Olasheni's Deal Hunter")
        gr.Markdown("Run a hunt → see the best deal; send it to your phone or pick any row to send.")
        hero = gr.Markdown(value="")
        run_btn = gr.Button("Run hunt", variant="primary")
        table = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], label="All from this hunt", wrap=True, row_count=8)
        row_status = gr.Textbox(label="Row action", interactive=False)

        run_btn.click(fn=run_and_store, outputs=[hero, table, results_state, row_status])
        table.select(fn=send_best_from_table, inputs=[results_state], outputs=[row_status])

    return ui

demo = ui_hunt()
demo.launch(inbrowser=True)